# ML workflow dashboard

A visual overview of the models trained in `force_correction_ml`. It explores the kind of tooling asked for in [FOWT-ML#10](https://github.com/hybridlabs-nl/FOWT-ML/issues/10) (visualising model training and comparison, PyCaret-style plots, dashboards). Nothing here is part of FOWT-ML; what, if anything, gets ported is decided later.

**Run `force_correction_ml` first.** Its last cell saves the fitted models, scores and hold-out rows to `inventory/ml_results/`, which this notebook reads. Nothing is re-trained here.

**Caveat:** the hold-out set is currently a *shuffled row split*, so near-identical neighbouring samples sit in both the training and hold-out data. All scores below are optimistic until the workflow splits by experiment.

Sections:
1. Run summary
2. Model comparison
3. Inspecting one model: prediction error and residuals
4. Time series for one experiment (interactive)

TODO: there is a correct_force func in utils.py; use for some comparisons as well?

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from matplotlib.colors import LinearSegmentedColormap
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

In [ ]:
# settings
RESULTS_DIR = Path("../inventory/ml_results")
MODEL_TO_INSPECT = None  # e.g. "XGBoostRegression"; None uses the pipeline's best model
EXPERIMENT_TO_INSPECT = (
    None  # recording number, e.g. 1653; None uses the first in the hold-out set
)

In [ ]:
# style: categorical slots follow the entity (model), never its rank
CATEGORICAL = [
    "#2a78d6",
    "#eb6834",
    "#1baf7a",
    "#eda100",
    "#e87ba4",
    "#008300",
    "#6250d6",
    "#e34948",
]
INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
SURFACE = "#fcfcfb"
GRID = "#e4e3de"

# one-hue ramp for magnitudes (heatmaps, hexbins), light = low
BLUES = LinearSegmentedColormap.from_list(
    "blues", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"]
)

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": "#bdbcb6",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "text.color": INK,
        "axes.labelcolor": INK_SECONDARY,
        "axes.titlecolor": INK,
        "axes.titlesize": 10,
        "axes.titlelocation": "left",
        "xtick.color": INK_SECONDARY,
        "ytick.color": INK_SECONDARY,
        "font.size": 9,
        "lines.linewidth": 1.5,
        "legend.frameon": False,
        "figure.dpi": 110,
    }
)

## 1 Run summary

In [ ]:
# load results saved by `force_correction_ml`
models = joblib.load(RESULTS_DIR / "models.joblib")
scores = pd.read_csv(RESULTS_DIR / "scores.csv", index_col="model")
run_info = json.loads((RESULTS_DIR / "run_info.json").read_text())
test_data = pd.read_parquet(RESULTS_DIR / "test_data.parquet")

predictors = run_info["predictors"]
targets = run_info["targets"]
OUTPUT_LABELS = ["Fx [N]", "Fy [N]", "Fz [N]", "Mx [Nm]", "My [Nm]", "Mz [Nm]"]

# colours are assigned in config order, so they don't change when the ranking does
MODEL_NAMES = list(models)
COLOURS = {name: CATEGORICAL[i] for i, name in enumerate(MODEL_NAMES)}

model_name = MODEL_TO_INSPECT or run_info["best_model"]
recording = EXPERIMENT_TO_INSPECT or int(test_data["recording_number"].iloc[0])

In [ ]:
pd.Series(
    {
        "results saved": run_info["created"],
        "data folder": Path(run_info["data_folder"]).name,
        "recordings": ", ".join(map(str, run_info["recordings"])),
        "rows (train/val, test)": f"{run_info['n_trainval']:,}, {run_info['n_test']:,}",
        "hold-out split": run_info["holdout_split"],
        "pipeline split": run_info["pipeline_split"],
        "models": ", ".join(MODEL_NAMES),
        "best model (pipeline score)": run_info["best_model"],
        "model inspected below": model_name,
        "experiment inspected below": recording,
    },
    name="value",
).to_frame()

In [ ]:
# hyperparameters as configured (PyCaret's 'parameter' view)
pd.DataFrame(run_info["model_names"]).T

## 2 Model comparison

All models are scored on the same hold-out rows, per output.

In [ ]:
def per_output_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    """Return RMSE, MAE and R² for each of the six outputs."""
    return pd.DataFrame(
        {
            "RMSE": root_mean_squared_error(y_true, y_pred, multioutput="raw_values"),
            "MAE": mean_absolute_error(y_true, y_pred, multioutput="raw_values"),
            "R2": r2_score(y_true, y_pred, multioutput="raw_values"),
        },
        index=pd.Index(OUTPUT_LABELS, name="output"),
    )

In [ ]:
# predictions on the hold-out rows
X = test_data[predictors].to_numpy()
Y = test_data[targets].to_numpy()
predictions = {name: model.predict(X) for name, model in models.items()}

metrics = pd.concat(
    {name: per_output_metrics(Y, predictions[name]) for name in MODEL_NAMES},
    names=["model"],
)

### 2.1 Scores reported by the pipeline

These come from FOWT-ML's own train/test split *inside* the hold-out-free data, averaged over the six outputs. Section 2.2 onwards uses the separate hold-out set, per output.

In [ ]:
scores

### 2.2 Hold-out metrics per output (table view)

In [ ]:
metrics["R2"].unstack().loc[MODEL_NAMES, OUTPUT_LABELS].round(3)

In [ ]:
metrics["RMSE"].unstack().loc[MODEL_NAMES, OUTPUT_LABELS].round(4)

### 2.3 R² per model and output

In [ ]:
def plot_heatmap(
    table: pd.DataFrame, ax: plt.Axes, title: str, vmin: float = 0.0, vmax: float = 1.0
) -> None:
    """Draw an annotated heatmap; values outside the colour range keep their true value as text."""
    shown = table.clip(vmin, vmax)
    ax.imshow(shown.to_numpy(), cmap=BLUES, vmin=vmin, vmax=vmax, aspect="auto")

    # annotate every cell, with ink chosen for contrast against the cell's shade
    for i in range(table.shape[0]):
        for j in range(table.shape[1]):
            shade = (shown.iat[i, j] - vmin) / (vmax - vmin)
            ax.text(
                j,
                i,
                f"{table.iat[i, j]:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if shade > 0.55 else INK,
            )

    ax.set_xticks(range(table.shape[1]), table.columns)
    ax.set_yticks(range(table.shape[0]), table.index)
    ax.tick_params(length=0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title(title)


r2_table = metrics["R2"].unstack().loc[MODEL_NAMES, OUTPUT_LABELS]
fig, ax = plt.subplots(figsize=(7.5, 0.5 * len(MODEL_NAMES) + 1))
plot_heatmap(r2_table, ax, "Hold-out R² per output (colour clipped to 0–1)")
fig.tight_layout()
plt.show()

## 3 Inspecting one model

PyCaret-style 'error' and 'residuals' plots for the model chosen in the settings cell (`MODEL_TO_INSPECT`). Hexbins are used because there are tens of thousands of points.

In [ ]:
y_pred = predictions[model_name]
print(f"Inspecting: {model_name}")


def robust_limits(*arrays: np.ndarray, pct: float = 0.1) -> tuple[float, float]:
    """Return shared axis limits that ignore the most extreme 0.1 % at each end."""
    values = np.concatenate([a.ravel() for a in arrays])
    return tuple(np.percentile(values, [pct, 100 - pct]))

### 3.1 Prediction error (predicted versus measured)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 6.2))
for j, ax in enumerate(axes.ravel()):
    lo, hi = robust_limits(Y[:, j], y_pred[:, j])
    ax.hexbin(
        Y[:, j],
        y_pred[:, j],
        gridsize=60,
        cmap=BLUES,
        mincnt=1,
        bins="log",
        extent=(lo, hi, lo, hi),
    )
    ax.plot([lo, hi], [lo, hi], color=INK_SECONDARY, linestyle="--", linewidth=1)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect("equal")
    ax.set_title(
        f"{OUTPUT_LABELS[j]}   R² = {metrics.loc[(model_name, OUTPUT_LABELS[j]), 'R2']:.3f}"
    )
    ax.set_xlabel("measured")
    ax.set_ylabel("predicted")
fig.suptitle(
    f"Prediction error, {model_name} (dashed: perfect prediction)",
    x=0.01,
    ha="left",
    fontsize=11,
)
fig.tight_layout()
plt.show()

### 3.2 Residuals versus predicted

Structure here (curvature, fanning, offsets) points to effects the model hasn't captured.

In [ ]:
residuals = Y - y_pred
fig, axes = plt.subplots(2, 3, figsize=(10, 5.6))
for j, ax in enumerate(axes.ravel()):
    x_lo, x_hi = robust_limits(y_pred[:, j])
    r_lo, r_hi = robust_limits(residuals[:, j])
    r_max = max(abs(r_lo), abs(r_hi))
    ax.hexbin(
        y_pred[:, j],
        residuals[:, j],
        gridsize=60,
        cmap=BLUES,
        mincnt=1,
        bins="log",
        extent=(x_lo, x_hi, -r_max, r_max),
    )
    ax.axhline(0, color=INK_SECONDARY, linestyle="--", linewidth=1)
    ax.set_xlim(x_lo, x_hi)
    ax.set_ylim(-r_max, r_max)
    ax.set_title(OUTPUT_LABELS[j])
    ax.set_xlabel("predicted")
    ax.set_ylabel("measured − predicted")
fig.suptitle(f"Residuals, {model_name}", x=0.01, ha="left", fontsize=11)
fig.tight_layout()
plt.show()

### 3.3 Residual distributions, all models

A shifted peak is a bias (e.g. a constant offset); a wide one is scatter.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10, 5.4))
for j, ax in enumerate(axes.ravel()):
    res = {name: Y[:, j] - predictions[name][:, j] for name in MODEL_NAMES}
    lo, hi = robust_limits(*res.values(), pct=1)
    bins = np.linspace(lo, hi, 80)
    for name in MODEL_NAMES:
        ax.hist(
            res[name],
            bins=bins,
            density=True,
            histtype="step",
            linewidth=1.5,
            color=COLOURS[name],
            label=name,
        )
    ax.axvline(0, color=INK_SECONDARY, linestyle=":", linewidth=1)
    ax.set_title(OUTPUT_LABELS[j])
    ax.set_xlabel("measured − predicted")
    ax.set_yticks([])
fig.legend(
    *axes[0, 0].get_legend_handles_labels(),
    loc="upper center",
    ncol=len(MODEL_NAMES),
    bbox_to_anchor=(0.5, 1.04),
)
fig.tight_layout()
plt.show()

## 4 Time series for one experiment (interactive)

Measured load against the inspected model, for the hold-out rows of one experiment (`EXPERIMENT_TO_INSPECT`). With the shuffled split, the hold-out rows are a random ~20 % of each experiment, so the trace is a thinned version of the full signal.

Drag to zoom, use the range slider at the bottom to slide along the experiment, and double-click to reset. All six panels share the time axis. The other models start hidden: click their legend entries to add them.

In [ ]:
rows = (test_data["recording_number"] == recording).to_numpy()
order = np.argsort(test_data.loc[rows, "time"].to_numpy())
t = test_data.loc[rows, "time"].to_numpy()[order]

fig = make_subplots(
    rows=len(OUTPUT_LABELS), cols=1, shared_xaxes=True, vertical_spacing=0.012
)
series = [("measured", Y, INK)] + [
    (name, predictions[name], COLOURS[name]) for name in MODEL_NAMES
]
for j, label in enumerate(OUTPUT_LABELS):
    for name, values, colour in series:
        fig.add_trace(
            go.Scatter(
                x=t,
                y=values[rows, j][order],
                name=name,
                # one legend entry per series, toggling it in all six panels
                legendgroup=name,
                showlegend=(j == 0),
                visible=True if name in ("measured", model_name) else "legendonly",
                mode="lines",
                line={"color": colour, "width": 1 if name == "measured" else 1.4},
                hovertemplate=f"{name}: %{{y:.3g}}<extra></extra>",
            ),
            row=j + 1,
            col=1,
        )
    fig.update_yaxes(title_text=label, row=j + 1, col=1)

fig.update_xaxes(showgrid=True, gridcolor=GRID, linecolor="#bdbcb6")
fig.update_yaxes(showgrid=True, gridcolor=GRID, zeroline=False, linecolor="#bdbcb6")
fig.update_xaxes(
    title_text="time [s]",
    rangeslider={"visible": True, "thickness": 0.05},
    row=len(OUTPUT_LABELS),
    col=1,
)
fig.update_layout(
    title={"text": f"Recording {recording}, hold-out rows", "x": 0.01},
    height=1050,
    hovermode="x unified",
    paper_bgcolor=SURFACE,
    plot_bgcolor=SURFACE,
    font={"color": INK, "size": 11},
    legend={"orientation": "h", "x": 0, "y": 1.04},
    margin={"l": 70, "r": 20, "t": 90, "b": 40},
)
fig.show()